In [28]:
import numpy as np
import pandas as pd

MATRIX_PATH = "GSE157103_series_matrix.txt"
TPM_PATH = "GSE157103_genes.tpm.tsv"
MIN_TPM = 1          # a gene counts as "expressed" in a sample if TPM >= this
MIN_FRACTION = 0.10  # keep genes expressed in at least 10% of samples
TOP_N_GENES = 2000   # keep the N most variable genes (None = keep all)

# 1. READ CLINICAL DATA
patients, gsm_ids, short_ids = [], [], []
info = {}
with open(MATRIX_PATH) as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        label = parts[0]
        values = [v.strip('"') for v in parts[1:]]
        if label == "!Sample_title":
            patients = values
        elif label == "!Sample_geo_accession":
            gsm_ids = values
        elif label == "!Sample_description" and not short_ids:
            short_ids = values            # first description line: C1, C2, ... NC26
        elif label == "!Sample_characteristics_ch1":
            name = values[0].split(":")[0]
            if name in ("icu", "Sex", "age (years)"):
                info[name] = [v.split(":", 1)[1].strip() for v in values]

clinical = pd.DataFrame(info, index=patients)          # patients as rows for now
clinical = clinical.rename(columns={"Sex": "gender", "age (years)": "age"})
clinical["covid"] = [0 if p.startswith("NONCOVID") else 1 for p in patients]

# 2. READ EXPRESSION DATA
expr = pd.read_csv(TPM_PATH, sep="\t", index_col=0)    # first column = gene symbols
print("Expression table:", expr.shape)
print("Expression columns look like:", list(expr.columns[:3]))

# rename expression columns to patient names: try each naming scheme, keep the best
candidates = {
    "short id (C1, NC1...)": dict(zip(short_ids, patients)),
    "GSM id": dict(zip(gsm_ids, patients)),
    "patient name": {p: p for p in patients},
}
best = max(candidates, key=lambda k: sum(c in candidates[k] for c in expr.columns))
mapping = candidates[best]
print("Matching columns by:", best)

matched = [c for c in expr.columns if c in mapping]
if len(matched) < len(patients):
    print("WARNING: only", len(matched), "of", len(patients), "samples matched")
expr = expr[matched].rename(columns=mapping)
expr = expr[clinical.index]                  # same patient order as the clinical table

# 3. CLEAN CLINICAL DATA
clinical["age"] = clinical["age"].replace(">89", "90")
clinical["age"] = pd.to_numeric(clinical["age"], errors="coerce")   # ":" -> NaN
clinical["gender"] = clinical["gender"].map({"male": 1, "female": 0})  # "unknown" -> NaN
clinical["icu"] = clinical["icu"].map({"yes": 1, "no": 0})
clinical["age"] = clinical["age"].fillna(clinical["age"].median())
clinical["gender"] = clinical["gender"].fillna(clinical["gender"].mode()[0])
clinical = clinical[["icu", "gender", "age", "covid"]]

# 4. CLEAN EXPRESSION DATA
n_start = expr.shape[0]
expr = expr.groupby(expr.index).mean()                      # merge duplicate gene symbols
expr = expr[expr.index.notna() & (expr.index != "")]        # drop rows with no symbol
expr = expr[(expr >= MIN_TPM).mean(axis=1) >= MIN_FRACTION] # drop barely-expressed genes
expr = np.log2(expr + 1)                                    # log2 transform
if TOP_N_GENES is not None and expr.shape[0] > TOP_N_GENES:
    expr = expr.loc[expr.var(axis=1).nlargest(TOP_N_GENES).index]

final = pd.concat([clinical.T, expr])
name_to_short = dict(zip(patients, short_ids))
final = final.rename(columns=name_to_short)
pd.Series(name_to_short, name="short_id").to_csv("patient_id_key.csv")

# z-scored version (each gene: mean 0, std 1 across patients)
genes = expr.index
final_scaled = final.copy()
g = expr.T
final_scaled.loc[genes] = ((g - g.mean()) / g.std().replace(0, 1)).T

final.to_csv("preprocessed_log2tpm.csv")            # rows = features, columns = patients
final_scaled.to_csv("preprocessed_scaled.csv")
# patients-as-rows version, the layout most ML tools expect
final.T.to_csv("preprocessed_log2tpm_patients_as_rows.csv")

final.head(19472)

Expression table: (19472, 126)
Expression columns look like: ['C1', 'C2', 'C3']
Matching columns by: short id (C1, NC1...)
Missing clinical values:
 age       1
gender    1
icu       0
covid     0
dtype: int64 



,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,...,NC17,NC18,NC19,NC20,NC21,NC22,NC23,NC24,NC25,NC26
icu,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
gender,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
age,39.000000,63.000000,33.000000,49.000000,49.000000,63.000000,38.000000,78.000000,64.000000,62.000000,...,84.000000,88.000000,66.000000,62.000000,71.000000,63.000000,42.000000,32.000000,62.000000,36.000000
covid,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RPS4Y1,7.707290,8.021091,6.900021,8.839172,7.427271,8.341408,0.000000,7.072427,0.000000,6.920412,...,0.000000,7.626658,0.042644,0.000000,9.197241,8.785518,0.000000,0.000000,9.382754,7.181798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ARFGEF1,4.641546,4.429616,3.938286,4.602291,4.329841,4.021480,4.595742,5.068671,4.484138,4.365972,...,4.384741,3.049631,3.244887,3.787641,2.662205,1.851999,2.395063,3.426265,1.584963,1.891419
SIGLEC5,7.414728,6.648322,7.552900,5.796494,7.090642,6.404290,5.967399,6.150966,6.620586,5.204376,...,7.258707,6.694323,6.577429,5.911931,5.150966,4.531693,6.245648,6.271463,4.359662,5.638943
EIF4B,6.652056,7.534030,5.364572,7.917730,6.620146,7.326250,7.769111,7.229203,7.480265,6.558880,...,7.201046,6.052025,5.317594,6.395748,6.537141,5.768184,5.608809,7.351028,5.200457,4.376429
RCBTB1,4.397118,3.533563,2.929791,3.928844,3.624101,3.114367,3.794936,3.646163,3.518535,3.025029,...,3.664483,2.653060,2.124328,2.298658,1.914565,1.475085,1.871844,3.072106,1.201634,1.000000
